# Phase 4: Directed Linguistic Probes

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Goal:** Test whether the PT-specific features identified in Phase 3 correspond to concrete Portuguese linguistic phenomena.

**What this notebook does:**
1. Loads model, SAE, and Phase 3 results (150 selected features)
2. Defines **linguistic probes** (minimal pairs) for 6 phenomena:
   - Gender agreement
   - Nominal agreement
   - Verbal conjugation
   - Crase
   - Personal infinitive
   - Clitic ordering
3. Defines **textual register probes** (legal, scientific, colloquial, journalistic)
4. Processes all probes through the model+SAE and measures differential activation
5. Maps features → phenomena and generates visualizations

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4)
- The `fase3_results.pt` file from Phase 3 (or individual checkpoints)
- Estimated runtime: ~20-30 min on T4

In [1]:
!pip install sae-lens transformer-lens -q
print()
print("Installation complete. Restart the runtime before continuing:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Depois, pule esta célula e execute a partir da próxima.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.2/195.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 10
from tqdm.auto import tqdm
import time
import os
import json

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('Sem GPU detectada.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4 (15.6 GB)
Device: cuda
PyTorch: 2.10.0+cu128


In [3]:
from huggingface_hub import login
login()

In [4]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Carregando Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Modelo: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Carregando SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

Carregando Gemma 2 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
Modelo: gemma-2-2b | Layers: 26 | d_model: 2304

Carregando SAE: layer_13/width_16k/canonical...


layer_13/width_16k/average_l0_84/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

SAE: 16384 features | d_in: 2304 | hook: blocks.13.hook_resid_post


/tmp/ipykernel_1557/3908203106.py:17: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [18]:
SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./checkpoints/ for Colab)
MIN_ACTS = 100
LSI_THRESHOLD = 0.3
N_SELECT = 50

fase3_path = SAVE_DIR + "fase3_results.pt"
if not os.path.exists(fase3_path):
    fase3_path = "fase3_results.pt"

if os.path.exists(fase3_path):
    print(f"Loading Phase 3 results: {fase3_path}")
    fase3 = torch.load(fase3_path, map_location="cpu")
    selected_pt = fase3["selected"]["pt"]
    selected_cross = fase3["selected"]["cross"]
    selected_en = fase3["selected"]["en"]
    all_selected = fase3["selected"]["all"]
    lsi_combined = fase3["lsi"]["combined"]
    lsi_wiki = fase3["lsi"]["wiki"]
    lsi_web = fase3["lsi"]["web"]
    pt_both = fase3["masks"]["pt_both"]

else:
    print("fase3_results.pt not found. Recomputing from individual checkpoints...")

    ckpt_files = ["stats_wiki_pt.pt", "stats_wiki_en.pt", "stats_mc4_pt.pt", "stats_c4_en.pt"]
    stats = {}
    for fname in ckpt_files:
        path = SAVE_DIR + fname
        if not os.path.exists(path):
            path = fname
        if not os.path.exists(path):
            raise FileNotFoundError(f"Checkpoint não encontrado: {fname}. Coloque em {SAVE_DIR} ou no diretório atual.")
        key = fname.replace("stats_", "").replace(".pt", "")
        stats[key] = torch.load(path, map_location="cpu")
        print(f"  Carregado: {fname} ({stats[key]['total_tokens']:,} tokens)")

    stats_wiki_pt = stats["wiki_pt"]
    stats_wiki_en = stats["wiki_en"]
    stats_mc4_pt = stats["mc4_pt"]
    stats_c4_en = stats["c4_en"]

    freq_wiki_pt = stats_wiki_pt["counts"] / stats_wiki_pt["total_tokens"]
    freq_wiki_en = stats_wiki_en["counts"] / stats_wiki_en["total_tokens"]
    freq_mc4_pt = stats_mc4_pt["counts"] / stats_mc4_pt["total_tokens"]
    freq_c4_en = stats_c4_en["counts"] / stats_c4_en["total_tokens"]

    lsi_wiki = (freq_wiki_pt - freq_wiki_en) / (freq_wiki_pt + freq_wiki_en + 1e-10)
    lsi_web = (freq_mc4_pt - freq_c4_en) / (freq_mc4_pt + freq_c4_en + 1e-10)
    lsi_combined = (lsi_wiki + lsi_web) / 2

    total_counts = stats_wiki_pt["counts"] + stats_wiki_en["counts"] + stats_mc4_pt["counts"] + stats_c4_en["counts"]
    active = total_counts >= MIN_ACTS

    pt_wiki = (lsi_wiki > LSI_THRESHOLD) & active
    pt_web = (lsi_web > LSI_THRESHOLD) & active
    pt_both = pt_wiki & pt_web

    lsi_pt_rank = lsi_combined.clone()
    lsi_pt_rank[~active] = -2
    _, top_pt_idx = lsi_pt_rank.topk(N_SELECT)
    selected_pt = top_pt_idx.tolist()

    lsi_en_rank = lsi_combined.clone()
    lsi_en_rank[~active] = 2
    _, top_en_idx = lsi_en_rank.topk(N_SELECT, largest=False)
    selected_en = top_en_idx.tolist()

    total_freq = freq_wiki_pt + freq_wiki_en + freq_mc4_pt + freq_c4_en
    cross_score = lsi_combined.abs().clone()
    cross_score[~active] = 999
    min_cross_freq = total_freq[active].median()
    cross_score[total_freq < min_cross_freq] = 999
    _, top_cross_idx = cross_score.topk(N_SELECT, largest=False)
    selected_cross = top_cross_idx.tolist()

    all_selected = selected_pt + selected_cross + selected_en
    print(f"\n  Seleção recomputada com sucesso!")

print(f"\nFeatures PT: {len(selected_pt)}")
print(f"Features cross-lingual: {len(selected_cross)}")
print(f"Features EN: {len(selected_en)}")
print(f"Total: {len(all_selected)}")

all_feature_indices = torch.tensor(all_selected)
pt_feature_indices = torch.tensor(selected_pt)
print(f"Features carregadas com sucesso.")

fase3_results.pt não encontrado. Recomputando a partir dos checkpoints individuais...
  Carregado: stats_wiki_pt.pt (4,999,936 tokens)
  Carregado: stats_wiki_en.pt (4,999,936 tokens)
  Carregado: stats_mc4_pt.pt (4,999,936 tokens)
  Carregado: stats_c4_en.pt (4,999,936 tokens)

  Seleção recomputada com sucesso!

Features PT: 50
Features cross-lingual: 50
Features EN: 50
Total: 150
Features carregadas com sucesso.


---
# Directed Linguistic Probes

For each phenomenon, we build **minimal pairs**: sentences that differ only in the linguistic aspect being tested.

| Phenomenon | Probe | Example |
|---|---|---|
| Gender agreement | Masc. vs fem. | "o menino bonito" vs "a menina bonita" |
| Nominal agreement | Correct vs incorrect | "as casas grandes" vs "as casas grande" |
| Verbal conjugation | Person variation | "eu corro / tu corres / ele corre" |
| Crase | With vs without | "vou à escola" vs "vou a escola" |
| Personal infinitive | Inflected vs not | "para nós fazermos" vs "para nós fazer" |
| Clitic ordering | Enclisis vs proclisis | "diga-me" vs "me diga" |

**Method:**
1. Process each text through Gemma 2 2B + SAE
2. Extract the maximum activation of each feature per text
3. Compute the **differential activation** between pair members
4. Features with high differential → sensitive to the phenomenon

In [6]:
PROBES = {
    "genero": {
        "name": "Concordância de Gênero",
        "type": "pairs",
        "pairs": [
            ("O menino bonito brincava no parque.", "A menina bonita brincava no parque."),
            ("O professor alto entrou na sala de aula.", "A professora alta entrou na sala de aula."),
            ("O gato preto dormia tranquilo no sofá.", "A gata preta dormia tranquila no sofá."),
            ("O aluno aplicado foi aprovado no exame.", "A aluna aplicada foi aprovada no exame."),
            ("O médico brasileiro foi premiado ontem.", "A médica brasileira foi premiada ontem."),
            ("O escritor famoso publicou um novo livro.", "A escritora famosa publicou um novo livro."),
            ("O cantor jovem apresentou-se no festival.", "A cantora jovem apresentou-se no festival."),
            ("O diretor responsável assinou o documento.", "A diretora responsável assinou o documento."),
        ],
        "labels": ("masculino", "feminino"),
    },
    "concordancia": {
        "name": "Concordância Nominal",
        "type": "pairs",
        "pairs": [
            ("As casas grandes foram vendidas rapidamente.", "As casas grande foram vendida rapidamente."),
            ("Os meninos bonitos correram pelo campo.", "Os meninos bonito correram pelo campo."),
            ("As flores vermelhas enfeitavam todo o jardim.", "As flores vermelha enfeitavam todo o jardim."),
            ("Os livros antigos estavam arrumados na estante.", "Os livros antigo estavam arrumados na estante."),
            ("As ruas estreitas dificultavam o trânsito.", "As ruas estreita dificultavam o trânsito."),
            ("Os prédios altos dominavam a paisagem.", "Os prédios alto dominavam a paisagem."),
        ],
        "labels": ("correto", "incorreto"),
    },
    "conjugacao": {
        "name": "Conjugação Verbal",
        "type": "sets",
        "sets": [
            [
                "Eu corro todos os dias no parque.",
                "Tu corres todos os dias no parque.",
                "Ele corre todos os dias no parque.",
                "Nós corremos todos os dias no parque.",
                "Eles correm todos os dias no parque.",
            ],
            [
                "Eu falo português fluentemente.",
                "Tu falas português fluentemente.",
                "Ele fala português fluentemente.",
                "Nós falamos português fluentemente.",
                "Eles falam português fluentemente.",
            ],
            [
                "Eu como frutas pela manhã.",
                "Tu comes frutas pela manhã.",
                "Ele come frutas pela manhã.",
                "Nós comemos frutas pela manhã.",
                "Eles comem frutas pela manhã.",
            ],
            [
                "Eu escrevo cartas para os amigos.",
                "Tu escreves cartas para os amigos.",
                "Ele escreve cartas para os amigos.",
                "Nós escrevemos cartas para os amigos.",
                "Eles escrevem cartas para os amigos.",
            ],
        ],
        "labels": ["eu", "tu", "ele/ela", "nós", "eles/elas"],
    },
    "crase": {
        "name": "Crase",
        "type": "pairs",
        "pairs": [
            ("Vou à escola todos os dias.", "Vou a escola todos os dias."),
            ("Refiro-me à questão principal do debate.", "Refiro-me a questão principal do debate."),
            ("Ele chegou à cidade ontem à noite.", "Ele chegou a cidade ontem a noite."),
            ("À medida que o tempo passava, tudo mudava.", "A medida que o tempo passava, tudo mudava."),
            ("Dedicou-se à pesquisa científica.", "Dedicou-se a pesquisa científica."),
            ("Fomos à praia no fim de semana.", "Fomos a praia no fim de semana."),
            ("Ele fez referência à obra de Machado.", "Ele fez referência a obra de Machado."),
            ("Graças à chuva, a seca acabou.", "Graças a chuva, a seca acabou."),
        ],
        "labels": ("com_crase", "sem_crase"),
    },
    "infinitivo_pessoal": {
        "name": "Infinitivo Pessoal",
        "type": "pairs",
        "pairs": [
            ("É importante para nós fazermos o trabalho.", "É importante para nós fazer o trabalho."),
            ("Antes de eles saírem de casa, verificaram tudo.", "Antes de eles sair de casa, verificaram tudo."),
            ("Apesar de nós termos estudado bastante.", "Apesar de nós ter estudado bastante."),
            ("Convém vocês pensarem bem antes de decidir.", "Convém vocês pensar bem antes de decidir."),
            ("É hora de nós partirmos para a viagem.", "É hora de nós partir para a viagem."),
            ("Sem eles perceberem, o tempo passou.", "Sem eles perceber, o tempo passou."),
        ],
        "labels": ("flexionado", "não_flexionado"),
    },
    "cliticos": {
        "name": "Ordem dos Clíticos",
        "type": "pairs",
        "pairs": [
            ("Diga-me a verdade sobre o assunto.", "Me diga a verdade sobre o assunto."),
            ("Encontrou-se com o amigo na praça.", "Se encontrou com o amigo na praça."),
            ("Deram-lhe o prêmio de melhor aluno.", "Lhe deram o prêmio de melhor aluno."),
            ("Viu-o caminhando pela rua principal.", "O viu caminhando pela rua principal."),
            ("Apresentou-nos o novo projeto ontem.", "Nos apresentou o novo projeto ontem."),
            ("Contaram-me uma história interessante.", "Me contaram uma história interessante."),
            ("Entregou-lhes os documentos necessários.", "Lhes entregou os documentos necessários."),
        ],
        "labels": ("ênclise", "próclise"),
    },
}

PROBES_REGISTER = {
    "name": "Registro Textual",
    "register_labels": ["juridico", "cientifico", "coloquial", "jornalistico"],
    "topics": [
        {
            "topic": "Demissão de funcionários",
            "texts": {
                "juridico": "A referida empresa procedeu à rescisão contratual dos colaboradores, em conformidade com o disposto no artigo 477 da Consolidação das Leis do Trabalho, assegurando o pagamento integral das verbas rescisórias devidas no prazo legal.",
                "cientifico": "A organização implementou um processo de redução de quadro funcional, fenômeno amplamente documentado na literatura sobre reestruturação corporativa e suas consequências socioeconômicas para os trabalhadores afetados.",
                "coloquial": "A empresa mandou um monte de gente embora de uma vez, ninguém esperava. O pessoal ficou todo sem chão, foi uma loucura total.",
                "jornalistico": "A empresa anunciou nesta terça-feira a demissão de funcionários em meio à crise econômica que atinge o setor. A medida afeta cerca de cem trabalhadores da unidade principal.",
            },
        },
        {
            "topic": "Reprovação em exame",
            "texts": {
                "juridico": "O discente não logrou êxito na avaliação acadêmica, conforme atesta o registro institucional lavrado pela secretaria do estabelecimento de ensino, cabendo recurso no prazo de quinze dias úteis.",
                "cientifico": "O participante apresentou desempenho inferior ao limiar de aprovação no instrumento de avaliação aplicado, resultado consistente com os indicadores de dificuldade da prova e o perfil acadêmico observado.",
                "coloquial": "O cara bombou na prova, estudou pouco e se deu mal mesmo. Agora vai ter que fazer tudo de novo no semestre que vem.",
                "jornalistico": "Estudante reprova em exame e questiona critérios de avaliação adotados pela instituição. A universidade afirma que os padrões seguem as diretrizes nacionais de educação.",
            },
        },
        {
            "topic": "Descoberta científica",
            "texts": {
                "juridico": "A referida pesquisa, devidamente registrada no órgão competente, resultou na identificação de uma nova substância com propriedades terapêuticas comprovadas mediante laudo técnico pericial elaborado por profissionais habilitados.",
                "cientifico": "Os resultados obtidos neste estudo demonstram a existência de um composto molecular com atividade anti-inflamatória significativa, conforme indicado pelos ensaios in vitro e in vivo realizados com amostras controladas.",
                "coloquial": "Os cientistas acharam um negócio novo que pode ajudar a tratar inflamação. É tipo um remédio natural que ninguém conhecia ainda, bem legal.",
                "jornalistico": "Pesquisadores brasileiros anunciam a descoberta de uma nova substância com potencial anti-inflamatório. O estudo, publicado em revista internacional de alto impacto, pode abrir caminho para novos tratamentos.",
            },
        },
        {
            "topic": "Acidente de trânsito",
            "texts": {
                "juridico": "No dia dos fatos, o condutor do veículo automotor colidiu com o poste de iluminação pública, resultando em danos materiais de monta considerável, conforme laudo pericial acostado aos autos do processo.",
                "cientifico": "O evento de colisão veicular contra estrutura fixa apresentou dinâmica compatível com velocidade acima do limite regulamentar, conforme análise das marcas de frenagem e deformação estrutural do veículo.",
                "coloquial": "O cara bateu o carro no poste, fez a maior confusão. Sorte que não se machucou, mas o carro ficou destruído, acabou com tudo.",
                "jornalistico": "Um motorista perdeu o controle do veículo e colidiu com um poste na avenida principal na madrugada desta segunda-feira. Não houve feridos, mas o trânsito ficou interditado por duas horas.",
            },
        },
    ],
}

n_probes = sum(
    len(p["pairs"]) * 2 if p["type"] == "pairs" else sum(len(s) for s in p["sets"])
    for p in PROBES.values()
)
n_register = len(PROBES_REGISTER["topics"]) * len(PROBES_REGISTER["register_labels"])

print(f"Probes linguísticos: {n_probes} textos")
print(f"Probes de registro: {n_register} textos")
print(f"Total: {n_probes + n_register} textos a processar")
print()
for key, probe in PROBES.items():
    if probe["type"] == "pairs":
        print(f"  {probe['name']}: {len(probe['pairs'])} pares")
    else:
        print(f"  {probe['name']}: {len(probe['sets'])} conjuntos × {len(probe['sets'][0])} variantes")
print(f"  {PROBES_REGISTER['name']}: {len(PROBES_REGISTER['topics'])} tópicos × {len(PROBES_REGISTER['register_labels'])} registros")

Probes linguísticos: 90 textos
Probes de registro: 16 textos
Total: 106 textos a processar

  Concordância de Gênero: 8 pares
  Concordância Nominal: 6 pares
  Conjugação Verbal: 4 conjuntos × 5 variantes
  Crase: 8 pares
  Infinitivo Pessoal: 6 pares
  Ordem dos Clíticos: 7 pares
  Registro Textual: 4 tópicos × 4 registros


In [7]:
@torch.no_grad()
def process_text(model, sae, text, hook_name, feature_indices=None):
    """Process a single text and return per-feature activations."""
    tokens = model.tokenizer.encode(text, return_tensors="pt").to(device)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: n == hook_name)
    acts = cache[hook_name]
    feat_acts = sae.encode(acts)  # (1, seq_len, d_sae)

    max_acts = feat_acts[0].max(dim=0).values.cpu()   # (d_sae,)
    mean_acts = feat_acts[0].mean(dim=0).cpu()         # (d_sae,)

    decoded = [model.tokenizer.decode([t]) for t in tokens[0].tolist()]

    result = {
        "text": text,
        "tokens": decoded,
        "n_tokens": len(decoded),
        "max_acts_all": max_acts,
        "mean_acts_all": mean_acts,
    }

    if feature_indices is not None:
        fi = torch.tensor(feature_indices) if not isinstance(feature_indices, torch.Tensor) else feature_indices
        result["max_acts"] = max_acts[fi]
        result["mean_acts"] = mean_acts[fi]
        result["token_acts"] = feat_acts[0, :, fi].cpu()  # (seq_len, n_features)

    del cache, acts, feat_acts
    if device == "cuda":
        torch.cuda.empty_cache()

    return result

# Quick test
test = process_text(model, sae, "Vou à escola.", HOOK_NAME, all_feature_indices)
print(f"Teste OK: '{test['text']}' → {test['n_tokens']} tokens, {test['max_acts'].shape[0]} features")

Teste OK: 'Vou à escola.' → 5 tokens, 150 features


In [8]:
print("=" * 70)
print("PROCESSANDO PROBES LINGUÍSTICOS")
print("=" * 70)

probe_results = {}

for key, probe in PROBES.items():
    print(f"\n--- {probe['name']} ---")

    if probe["type"] == "pairs":
        results_a = []
        results_b = []
        for i, (text_a, text_b) in enumerate(probe["pairs"]):
            r_a = process_text(model, sae, text_a, HOOK_NAME, all_feature_indices)
            r_b = process_text(model, sae, text_b, HOOK_NAME, all_feature_indices)
            results_a.append(r_a)
            results_b.append(r_b)
            print(f"  Par {i+1}/{len(probe['pairs'])}: OK")

        probe_results[key] = {
            "type": "pairs",
            "results_a": results_a,
            "results_b": results_b,
            "labels": probe["labels"],
            "name": probe["name"],
        }

    elif probe["type"] == "sets":
        all_set_results = []
        for s_idx, text_set in enumerate(probe["sets"]):
            set_results = []
            for text in text_set:
                r = process_text(model, sae, text, HOOK_NAME, all_feature_indices)
                set_results.append(r)
            all_set_results.append(set_results)
            print(f"  Conjunto {s_idx+1}/{len(probe['sets'])}: OK")

        probe_results[key] = {
            "type": "sets",
            "results": all_set_results,
            "labels": probe["labels"],
            "name": probe["name"],
        }

print("\nTodos os probes linguísticos processados!")

PROCESSANDO PROBES LINGUÍSTICOS

--- Concordância de Gênero ---
  Par 1/8: OK
  Par 2/8: OK
  Par 3/8: OK
  Par 4/8: OK
  Par 5/8: OK
  Par 6/8: OK
  Par 7/8: OK
  Par 8/8: OK

--- Concordância Nominal ---
  Par 1/6: OK
  Par 2/6: OK
  Par 3/6: OK
  Par 4/6: OK
  Par 5/6: OK
  Par 6/6: OK

--- Conjugação Verbal ---
  Conjunto 1/4: OK
  Conjunto 2/4: OK
  Conjunto 3/4: OK
  Conjunto 4/4: OK

--- Crase ---
  Par 1/8: OK
  Par 2/8: OK
  Par 3/8: OK
  Par 4/8: OK
  Par 5/8: OK
  Par 6/8: OK
  Par 7/8: OK
  Par 8/8: OK

--- Infinitivo Pessoal ---
  Par 1/6: OK
  Par 2/6: OK
  Par 3/6: OK
  Par 4/6: OK
  Par 5/6: OK
  Par 6/6: OK

--- Ordem dos Clíticos ---
  Par 1/7: OK
  Par 2/7: OK
  Par 3/7: OK
  Par 4/7: OK
  Par 5/7: OK
  Par 6/7: OK
  Par 7/7: OK

Todos os probes linguísticos processados!


In [9]:
print("=" * 70)
print("ANÁLISE DIFERENCIAL POR FENÔMENO")
print("=" * 70)

n_all = len(all_selected)
differential_scores = {}

for key, pr in probe_results.items():
    if pr["type"] == "pairs":
        diffs = torch.zeros(n_all)
        for r_a, r_b in zip(pr["results_a"], pr["results_b"]):
            diffs += (r_a["max_acts"] - r_b["max_acts"])
        diffs /= len(pr["results_a"])
        differential_scores[key] = diffs

    elif pr["type"] == "sets":
        variances = torch.zeros(n_all)
        for set_results in pr["results"]:
            acts_stack = torch.stack([r["max_acts"] for r in set_results])  # (n_variants, n_features)
            variances += acts_stack.var(dim=0)
        variances /= len(pr["results"])
        differential_scores[key] = variances

print("\nScores computed for all phenomena.")
for key, scores in differential_scores.items():
    name = probe_results[key]["name"]
    if probe_results[key]["type"] == "pairs":
        print(f"  {name}: mean |diff| = {scores.abs().mean():.4f}, max |diff| = {scores.abs().max():.4f}")
    else:
        print(f"  {name}: mean var = {scores.mean():.4f}, max var = {scores.max():.4f}")

ANÁLISE DIFERENCIAL POR FENÔMENO

Scores calculados para todos os fenômenos.
  Concordância de Gênero: mean |diff| = 0.2249, max |diff| = 13.1447
  Concordância Nominal: mean |diff| = 0.1162, max |diff| = 2.3284
  Conjugação Verbal: mean var = 1.3956, max var = 69.7138
  Crase: mean |diff| = 0.2638, max |diff| = 9.4942
  Infinitivo Pessoal: mean |diff| = 0.2594, max |diff| = 10.7375
  Ordem dos Clíticos: mean |diff| = 0.2036, max |diff| = 3.4671


In [10]:
print("=" * 80)
print("TOP FEATURES SENSÍVEIS POR FENÔMENO")
print("=" * 80)

TOP_K = 10
feature_phenomenon_map = {}

for key, scores in differential_scores.items():
    pr = probe_results[key]
    name = pr["name"]
    print(f"\n{'—' * 80}")
    print(f"{name}")
    print(f"{'—' * 80}")

    if pr["type"] == "pairs":
        abs_scores = scores.abs()
        topk_vals, topk_local = abs_scores.topk(TOP_K)

        fmt = "{:<4} {:<8} {:<10} {:<10} {:<10} {:<12}"
        print(fmt.format("#", "Feature", "Diff", "|Diff|", "LSI comb", "Categoria"))
        print("-" * 70)

        for rank, (local_idx, val) in enumerate(zip(topk_local, topk_vals)):
            feat_id = all_selected[local_idx.item()]
            diff_val = scores[local_idx.item()].item()
            lsi_val = lsi_combined[feat_id].item()

            if feat_id in selected_pt:
                cat = "PT-específica"
            elif feat_id in selected_en:
                cat = "EN-específica"
            else:
                cat = "Cross-lingual"

            feature_phenomenon_map.setdefault(feat_id, []).append({
                "phenomenon": key,
                "name": name,
                "diff": diff_val,
                "rank": rank + 1,
            })

            direction = "→" + pr["labels"][0] if diff_val > 0 else "→" + pr["labels"][1]
            print(fmt.format(
                rank + 1, feat_id,
                f"{diff_val:+.4f}",
                f"{val.item():.4f}",
                f"{lsi_val:+.4f}",
                cat,
            ))
            print(f"       {direction}")

    elif pr["type"] == "sets":
        topk_vals, topk_local = scores.topk(TOP_K)
        fmt = "{:<4} {:<8} {:<10} {:<10} {:<12}"
        print(fmt.format("#", "Feature", "Variance", "LSI comb", "Categoria"))
        print("-" * 60)

        for rank, (local_idx, val) in enumerate(zip(topk_local, topk_vals)):
            feat_id = all_selected[local_idx.item()]
            lsi_val = lsi_combined[feat_id].item()

            if feat_id in selected_pt:
                cat = "PT-específica"
            elif feat_id in selected_en:
                cat = "EN-específica"
            else:
                cat = "Cross-lingual"

            feature_phenomenon_map.setdefault(feat_id, []).append({
                "phenomenon": key,
                "name": name,
                "variance": val.item(),
                "rank": rank + 1,
            })

            acts_per_person = []
            for set_results in pr["results"]:
                acts_stack = torch.stack([r["max_acts"] for r in set_results])
                acts_per_person.append(acts_stack[:, local_idx.item()])
            mean_per_person = torch.stack(acts_per_person).mean(dim=0)
            best_person = pr["labels"][mean_per_person.argmax().item()]

            print(fmt.format(
                rank + 1, feat_id,
                f"{val.item():.4f}",
                f"{lsi_val:+.4f}",
                cat,
            ))
            print(f"       Most active for: {best_person}")

print()
print("=" * 80)
print(f"RESUMO: {len(feature_phenomenon_map)} features mapeadas a fenômenos")
print("=" * 80)
n_pt_mapped = sum(1 for f in feature_phenomenon_map if f in selected_pt)
n_cross_mapped = sum(1 for f in feature_phenomenon_map if f in selected_cross)
n_en_mapped = sum(1 for f in feature_phenomenon_map if f in selected_en)
print(f"  PT-específicas: {n_pt_mapped}")
print(f"  Cross-lingual:  {n_cross_mapped}")
print(f"  EN-específicas: {n_en_mapped}")

TOP FEATURES SENSÍVEIS POR FENÔMENO

————————————————————————————————————————————————————————————————————————————————
Concordância de Gênero
————————————————————————————————————————————————————————————————————————————————
#    Feature  Diff       |Diff|     LSI comb   Categoria   
----------------------------------------------------------------------
1    1215     -13.1447   13.1447    +0.9483    PT-específica
       →feminino
2    15887    -5.7645    5.7645     +0.0008    Cross-lingual
       →feminino
3    1950     -2.4754    2.4754     +0.9410    PT-específica
       →feminino
4    4584     -1.5825    1.5825     +0.9953    PT-específica
       →feminino
5    138      -1.1562    1.1562     +0.9692    PT-específica
       →feminino
6    2294     +1.1257    1.1257     +0.9944    PT-específica
       →masculino
7    13526    +1.0846    1.0846     +0.0008    Cross-lingual
       →masculino
8    2817     -0.8278    0.8278     +0.9904    PT-específica
       →feminino
9    9166     -0.6512

In [11]:
phenomenon_keys = list(differential_scores.keys())
phenomenon_names = [probe_results[k]["name"] for k in phenomenon_keys]

n_pt = len(selected_pt)
heatmap_data = np.zeros((n_pt, len(phenomenon_keys)))

for j, key in enumerate(phenomenon_keys):
    scores = differential_scores[key]
    for i, feat_id in enumerate(selected_pt):
        local_idx = all_selected.index(feat_id)
        if probe_results[key]["type"] == "pairs":
            heatmap_data[i, j] = scores[local_idx].abs().item()
        else:
            heatmap_data[i, j] = scores[local_idx].item()

row_max = heatmap_data.max(axis=1, keepdims=True)
row_max[row_max == 0] = 1
heatmap_norm = heatmap_data / row_max

fig, ax = plt.subplots(figsize=(10, 16))
im = ax.imshow(heatmap_norm, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(len(phenomenon_names)))
ax.set_xticklabels(phenomenon_names, rotation=45, ha='right', fontsize=9)

ylabels = [f'F{fid} (LSI={lsi_combined[fid].item():+.2f})' for fid in selected_pt]
ax.set_yticks(range(n_pt))
ax.set_yticklabels(ylabels, fontsize=6)

ax.set_title('Feature Sensitivity to Linguistic Phenomena\n(Top-50 PT-Specific Features)', fontsize=12)
plt.colorbar(im, label='Normalized |differential|', shrink=0.6)
plt.tight_layout()
plt.savefig('fig_fase4_feature_phenomenon_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_fase4_feature_phenomenon_heatmap.png")

Salvo: fig_fase4_feature_phenomenon_heatmap.png


In [12]:
print("=" * 80)
print("TOKEN-BY-TOKEN ANALYSIS — MOST SENSITIVE FEATURES")
print("=" * 80)

for key in ["crase", "cliticos", "genero"]:
    pr = probe_results[key]
    scores = differential_scores[key]
    abs_scores = scores.abs()
    top_local = abs_scores.topk(3).indices

    print(f"\n{'═' * 80}")
    print(f"  {pr['name'].upper()}")
    print(f"{'═' * 80}")

    for local_idx in top_local:
        feat_id = all_selected[local_idx.item()]
        print(f"\n  Feature {feat_id} (LSI={lsi_combined[feat_id].item():+.4f}, diff={scores[local_idx.item()]:+.4f})")

        pair_idx = 0
        r_a = pr["results_a"][pair_idx]
        r_b = pr["results_b"][pair_idx]

        print(f"\n    {pr['labels'][0].upper()}: \"{r_a['text']}\"")
        acts_a = r_a["token_acts"][:, local_idx.item()]
        for t_idx, (tok, act) in enumerate(zip(r_a["tokens"], acts_a)):
            bar = "█" * int(act.item() * 5)
            if act.item() > 0.1:
                print(f"      [{t_idx:2d}] {tok:>15s}  {act.item():6.3f} {bar}")

        print(f"\n    {pr['labels'][1].upper()}: \"{r_b['text']}\"")
        acts_b = r_b["token_acts"][:, local_idx.item()]
        for t_idx, (tok, act) in enumerate(zip(r_b["tokens"], acts_b)):
            bar = "█" * int(act.item() * 5)
            if act.item() > 0.1:
                print(f"      [{t_idx:2d}] {tok:>15s}  {act.item():6.3f} {bar}")

ANÁLISE TOKEN-A-TOKEN — FEATURES MAIS SENSÍVEIS

════════════════════════════════════════════════════════════════════════════════
  CRASE
════════════════════════════════════════════════════════════════════════════════

  Feature 4584 (LSI=+0.9953, diff=+9.4942)

    COM_CRASE: "Vou à escola todos os dias."
      [ 2]               à  24.185 ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

    SEM_CRASE: "Vou a escola todos os dias."
      [ 2]               a  13.612 ████████████████████████████████████████████████████████████████████

  Feature 2294 (LSI=+0.9944, diff=+9.3714)

    COM_CRASE: "Vou à escola todos os dias."
      [ 2]               à   6.744 █████████████████████████████████

    SEM_CRASE: "Vou a escola todos os dias."

  Feature 5742 (LSI=+0.9108, diff=+7.3660)

    COM_CRASE: "Vou à escola todos os dias."
      [ 2]               à  14.602 ███████████████████████████████████████████████████████

---
# Textual Register Probes

We test whether the SAE captures **register/tone** differences (legal, scientific, colloquial, journalistic) for the same semantic content.

This is the basis for **tone feature steering**: if we find register-sensitive features, we can amplify them to control the generation style.

In [13]:
print("=" * 70)
print("PROCESSANDO PROBES DE REGISTRO TEXTUAL")
print("=" * 70)

register_labels = PROBES_REGISTER["register_labels"]
register_results = []

for t_idx, topic in enumerate(PROBES_REGISTER["topics"]):
    print(f"\n  Tópico {t_idx+1}: {topic['topic']}")
    topic_results = {}
    for reg in register_labels:
        text = topic["texts"][reg]
        r = process_text(model, sae, text, HOOK_NAME, all_feature_indices)
        topic_results[reg] = r
        print(f"    {reg}: OK ({r['n_tokens']} tokens)")
    register_results.append(topic_results)

print("\nProbes de registro processados!")

PROCESSANDO PROBES DE REGISTRO TEXTUAL

  Tópico 1: Demissão de funcionários
    juridico: OK (56 tokens)
    cientifico: OK (40 tokens)
    coloquial: OK (31 tokens)
    jornalistico: OK (36 tokens)

  Tópico 2: Reprovação em exame
    juridico: OK (40 tokens)
    cientifico: OK (36 tokens)
    coloquial: OK (30 tokens)
    jornalistico: OK (37 tokens)

  Tópico 3: Descoberta científica
    juridico: OK (44 tokens)
    cientifico: OK (41 tokens)
    coloquial: OK (32 tokens)
    jornalistico: OK (42 tokens)

  Tópico 4: Acidente de trânsito
    juridico: OK (44 tokens)
    cientifico: OK (40 tokens)
    coloquial: OK (35 tokens)
    jornalistico: OK (39 tokens)

Probes de registro processados!


In [14]:
print("=" * 80)
print("FEATURES SENSÍVEIS A REGISTRO TEXTUAL")
print("=" * 80)

n_all = len(all_selected)
register_labels = PROBES_REGISTER["register_labels"]
n_reg = len(register_labels)
n_topics = len(register_results)

register_acts = torch.zeros(n_topics, n_reg, n_all)
for t_idx, topic_results in enumerate(register_results):
    for r_idx, reg in enumerate(register_labels):
        register_acts[t_idx, r_idx] = topic_results[reg]["max_acts"]

mean_per_register = register_acts.mean(dim=0)  # (n_reg, n_all)
variance_per_feature = mean_per_register.var(dim=0)  # (n_all,)

topk_reg_vals, topk_reg_local = variance_per_feature.topk(15)

fmt = "{:<4} {:<8} {:<10} {:<10} {:<12} {:<10} {:<10} {:<10} {:<10}"
print(fmt.format("#", "Feature", "Variance", "LSI", "Categoria",
                  "Jurídico", "Científ.", "Coloq.", "Jornal."))
print("-" * 100)

register_feature_map = {}
for rank, (local_idx, val) in enumerate(zip(topk_reg_local, topk_reg_vals)):
    feat_id = all_selected[local_idx.item()]
    lsi_val = lsi_combined[feat_id].item()

    if feat_id in selected_pt:
        cat = "PT"
    elif feat_id in selected_en:
        cat = "EN"
    else:
        cat = "Cross"

    acts = mean_per_register[:, local_idx.item()]
    best_reg = register_labels[acts.argmax().item()]

    register_feature_map[feat_id] = {
        "variance": val.item(),
        "best_register": best_reg,
        "acts_per_register": {reg: acts[i].item() for i, reg in enumerate(register_labels)},
    }

    print(fmt.format(
        rank + 1, feat_id,
        f"{val.item():.4f}",
        f"{lsi_val:+.4f}",
        cat,
        f"{acts[0].item():.3f}",
        f"{acts[1].item():.3f}",
        f"{acts[2].item():.3f}",
        f"{acts[3].item():.3f}",
    ))
    print(f"       → Most active for: {best_reg}")

FEATURES SENSÍVEIS A REGISTRO TEXTUAL
#    Feature  Variance   LSI        Categoria    Jurídico   Científ.   Coloq.     Jornal.   
----------------------------------------------------------------------------------------------------
1    5880     127.8259   +0.9647    PT           17.374     31.727     4.074      16.597    
       → Mais ativa para: cientifico
2    2294     76.5407    +0.9944    PT           34.586     33.441     15.716     25.144    
       → Mais ativa para: juridico
3    12269    73.9909    +0.9742    PT           39.362     35.885     20.702     26.205    
       → Mais ativa para: juridico
4    1082     70.9350    +0.9566    PT           0.000      0.000      17.196     1.160     
       → Mais ativa para: coloquial
5    16194    28.4870    +0.9918    PT           0.000      12.917     4.789      6.395     
       → Mais ativa para: cientifico
6    16057    27.3462    +0.9965    PT           27.090     21.853     17.516     29.089    
       → Mais ativa para: jorn

In [15]:
top_reg_features = [all_selected[idx.item()] for idx in topk_reg_local[:15]]
data = np.zeros((len(top_reg_features), n_reg))
for i, feat_id in enumerate(top_reg_features):
    local_idx = all_selected.index(feat_id)
    data[i] = mean_per_register[:, local_idx].numpy()

row_max = data.max(axis=1, keepdims=True)
row_max[row_max == 0] = 1
data_norm = data / row_max

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(data_norm, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(n_reg))
ax.set_xticklabels([r.capitalize() for r in register_labels], fontsize=10)
ylabels = [f'F{fid} (LSI={lsi_combined[fid].item():+.2f})' for fid in top_reg_features]
ax.set_yticks(range(len(top_reg_features)))
ax.set_yticklabels(ylabels, fontsize=8)
ax.set_title('Register-Sensitive Features\n(normalized activation per register)', fontsize=12)
plt.colorbar(im, label='Normalized max activation', shrink=0.7)
plt.tight_layout()
plt.savefig('fig_fase4_register_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_fase4_register_heatmap.png")

Salvo: fig_fase4_register_heatmap.png


In [16]:
phenomenon_keys_all = list(differential_scores.keys()) + ["registro"]
phenomenon_names_all = [probe_results[k]["name"] for k in differential_scores.keys()] + ["Registro Textual"]

cat_labels = ["PT-específica", "Cross-lingual", "EN-específica"]
cat_features = [selected_pt, selected_cross, selected_en]
cat_colors = ["#e74c3c", "#3498db", "#2ecc71"]

fig, axes = plt.subplots(1, len(phenomenon_keys_all), figsize=(20, 5), sharey=True)

for j, (key, name) in enumerate(zip(phenomenon_keys_all, phenomenon_names_all)):
    ax = axes[j]

    if key == "registro":
        scores_all = variance_per_feature
    else:
        scores_all = differential_scores[key]
        if probe_results[key]["type"] == "pairs":
            scores_all = scores_all.abs()

    means = []
    stds = []
    for cat_feats in cat_features:
        local_idxs = [all_selected.index(f) for f in cat_feats]
        vals = scores_all[local_idxs]
        means.append(vals.mean().item())
        stds.append(vals.std().item())

    bars = ax.bar(range(3), means, yerr=stds, color=cat_colors, capsize=4, alpha=0.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["PT", "Cross", "EN"], fontsize=8)
    ax.set_title(name, fontsize=9)
    if j == 0:
        ax.set_ylabel("Mean sensitivity score")

plt.suptitle("Feature Sensitivity by Category and Phenomenon", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_fase4_category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_fase4_category_comparison.png")

Salvo: fig_fase4_category_comparison.png


In [ ]:
results_fase4 = {
    "probe_definitions": {k: {kk: vv for kk, vv in v.items() if kk != "type"} for k, v in PROBES.items()},
    "register_definitions": {
        "labels": register_labels,
        "topics": [t["topic"] for t in PROBES_REGISTER["topics"]],
    },
    "differential_scores": {k: v.cpu() for k, v in differential_scores.items()},
    "register_variance": variance_per_feature.cpu(),
    "register_mean_per_register": mean_per_register.cpu(),
    "feature_phenomenon_map": feature_phenomenon_map,
    "register_feature_map": register_feature_map,
    "all_selected": all_selected,
    "selected_pt": selected_pt,
    "selected_cross": selected_cross,
    "selected_en": selected_en,
}

save_path = SAVE_DIR + "fase4_probes_results.pt" if os.path.exists(SAVE_DIR) else "fase4_probes_results.pt"
torch.save(results_fase4, save_path)
print(f"Resultados salvos em: {save_path}")

print()
print("=" * 70)
print("ARQUIVOS GERADOS")
print("=" * 70)
print(f"  {save_path}")
print("  fig_fase4_feature_phenomenon_heatmap.png")
print("  fig_fase4_register_heatmap.png")
print("  fig_fase4_category_comparison.png")

print()
print("=" * 70)
print("PHASE 4 SUMMARY")
print("=" * 70)
print(f"  Probes linguísticos: 6 fenômenos processados")
print(f"  Probes de registro: {n_topics} tópicos × {n_reg} registros")
print(f"  Features mapeadas a fenômenos: {len(feature_phenomenon_map)}")
print(f"  Features sensíveis a registro: {len(register_feature_map)}")
print()

Resultados salvos em: ./checkpoints/fase4_probes_results.pt

ARQUIVOS GERADOS
  ./checkpoints/fase4_probes_results.pt
  fig_fase4_feature_phenomenon_heatmap.png
  fig_fase4_register_heatmap.png
  fig_fase4_category_comparison.png

RESUMO DA FASE 4
  Probes linguísticos: 6 fenômenos processados
  Probes de registro: 4 tópicos × 4 registros
  Features mapeadas a fenômenos: 33
  Features sensíveis a registro: 15

